# LLM Gateways

LLM Gateways are smart middleware layer that sits between your application and multiple LLM providers(OpenAI, Gemini, Claude)

LLM Gateways provide:
- Routing 
- Fallbacks
- Caching
- Rate Limiting
- Cost Tracking
- Observability

## Implementation

To implement LLM Gateways, we will be using
- <b>LiteLLM</b>
- <b>LangChain</b>

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY=os.getenv('GROQ_API_KEY')
GOOGLE_API_KEY=os.getenv('GOOGLE_API_KEY')
# os.getenv('OPENAI_API_KEY')

In [ ]:
import litellm
from litellm import completion

# litellm._turn_on_debug()

response = completion(
    model="groq/llama-3.3-70b-versatile", # Or any currently supported Groq model
    messages=[
        {"role": "user", "content": "What is RAG?"}
    ]
)



### Fallbacks

In [ ]:
response=completion(
    model='google/gemini-1.5-flash',
    messages=[{'role':'user','content':'explain RAG in single line'}],
    fallbacks=['groq/llama-3.3-70b-versatile',
               'gpt-4o-mini']
)

print(f'model that actually worked :{response.model}')

### Cost Analysis

In [ ]:
from litellm import completion,completion_cost

response=completion(
    model='groq/llama-3.3-70b-versatile',
    messages=[{'role':'user','content':'I want to analyze cost of my llm calls'}],
)

cost_analysis=completion_cost(completion_response=response,model='groq/llama-3.3-70b-versatile')

print('Response :',response.choices[0])

In [ ]:
print('Response :',response.choices[0].message.content)
print('Input Tokens :', response.usage.prompt_tokens)
print('Output Tokens :', response.usage.completion_tokens)
print('cost ', cost_analysis)

### Caching

<b>Callbacks</b>- callbacks are functions or handlers that get triggered automatically when certain events occur during execution

In [ ]:
## Resetting the llm global state and caching , this is not mandatory but just used to clear if anything is used prior
import litellm

litellm.callbacks=[]
litellm.success_callback=[]
litellm.failure_callback=[]
litellm._async_success_callback=[]
litellm._async_failure_callback=[]

litellm.caching=None


In [ ]:
import litellm
import time
from litellm import completion
from  litellm.caching import Cache,caching

litellm.cache=Cache(type='local')

prompt= 'what is llmlite'

start=time.time()

r1=completion(
    model='groq/llama-3.3-70b-versatile',
    messages=[{'role':'user','content':prompt}],
    caching=True

)

t1=time.time()-start
print('First call took',t1,'secs')

start=time.time()

r2=completion(
    model='groq/llama-3.3-70b-versatile',
    messages=[{'role':'user','content':prompt}],
    caching=True
)

t2=time.time()-start
print('Second call took',t2,'secs')

### Smart Routing
The right job for right model

we will use litellm's router to define routing rules

In [ ]:
import os
from litellm import Router

model_list=[
    {   
        'model_name':'fast-cheap',
        'litellm_params':{
            'model':'groq/llama-3.3-70b-versatile',
            'api_key':os.getenv('GROQ_API_KEY')
            }
    },
    {   
        'model_name':'smart-coding',
        'litellm_params':{
            'model':'gemini/gemini-2.5-flash',
            'api_key':os.getenv('GOOGLE_API_KEY')
            }
    },
    {   
        'model_name':'balanced',
        'litellm_params':{
            'model':'gemini/gemini-2.5-flash',
            'api_key':os.getenv('GOOGLE_API_KEY')
            }
    }
] 

router=Router(model_list=model_list)

fast_response = router.completion(
    model='fast-cheap',
    messages=[{'role':'user','content':'Summarize: AI is changing software'}]
)

code_response=router.completion(
    model='smart-coding',
    messages=[{'role':'user','content':'Write a python function to reverse a string'}]
)

print(fast_response)
print(code_response)


### Load Balancing Across Multiple API Keys

#### using smart shuffle

In [ ]:
from litellm import Router
import os
from collections import Counter

model_list = [
    {
        "model_name": "chat-model",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY")
        },
        "model_info":{"id":'Groq Model'}
    },
    {
        "model_name": "chat-model",
        "litellm_params": {
            "model": "gemini/gemini-2.5-flash",
            "api_key": os.getenv("GEMINI_API_KEY")
        },
        "model_info":{"id":'Gemini Model'}
    }
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

hits=Counter()
for i in range(8):
    r=router.completion(
        model="chat-model",
        messages=[{'role':'user','content':f'say \'OK\' {i}'}],
        max_tokens=5
    )
    hits[r._hidden_params.get('model_id','?')]+=1
    print(f'Request {i+1} ->{r._hidden_params.get('model_id','?')}')


In [ ]:
print('Distribution')
print(hits.most_common())
for k,v in hits.most_common():
    print(f"{k}:{''*v}({v})")

#### Using least busy

In [ ]:
from litellm import Router
import os
from collections import Counter

model_list = [
    {
        "model_name": "chat-model",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY")
        },
        "model_info":{"id":'Groq Model'}
    },
    {
        "model_name": "chat-model",
        "litellm_params": {
            "model": "gemini/gemini-2.5-flash",
            "api_key": os.getenv("GEMINI_API_KEY")
        },
        "model_info":{"id":'Gemini Model'}
    }
]

router = Router(
    model_list=model_list,
    routing_strategy="least-busy"
)

hits=Counter()

print(hits)

for i in range(8):
    r=router.completion(
        model="chat-model",
        messages=[{'role':'user','content':f'say \'OK\' {i}'}],
        max_tokens=5
    )
    print(r._hidden_params)
    hits[r._hidden_params.get('model_id','?')]+=1
    print(hits)
    print(f'Request {i+1} ->{r._hidden_params.get('model_id','?')}')


# Integrating Gateways with Langchain

In [ ]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm= ChatLiteLLM(model='gemini/gemini-2.5-flash')

prompt=ChatPromptTemplate.from_messages([ ('system','you are a helpful ai assistant'),
    ('user',"{question}")])

chain= prompt | llm | StrOutputParser()

answer=chain.invoke({'question':'Give about rivers in 3 points'})